# RMSC-Net: Robust Multi-Scale Consistency Network for Dense Crowd Counting
### End-to-End Google Colab Notebook
**Paper:** *RMSC-Net: A Lightweight Robust Multi-Scale Consistency Network for Dense Crowd Counting under Environmental and Adversarial Perturbations*

This notebook provides a complete workflow:
1. Environment setup & dependency installation
2. Google Drive mount & checkpoint management
3. ShanghaiTech Part A dataset preparation — **downloaded automatically** (Kaggle via `kagglehub`)
4. Model instantiation & parameter analysis
5. **Stage I** — Clean curriculum training (100 epochs)
6. **Stage II** — Consistency training with corruptions (100 epochs)
7. **Stage III** — Adversarial training (50 epochs)
8. Clean evaluation (MAE, RMSE, NAE, GAME)
9. Corruption robustness evaluation (12 types × 5 severities)
10. Adversarial robustness evaluation (FGSM, PGD, BIM)
11. XAI visualisations (gradient saliency, DAA maps, error maps)
12. Ablation study runner
13. Statistical validation & results dashboard

> **Everything is stored on Google Drive** under `MyDrive/RMSC-Net/` (checkpoints, TensorBoard logs, every metric as CSV/JSON, every figure, exported models), with a running `results/summary.json` of all headline numbers.

> **Accuracy settings (Section 2):** `ENCODER = "mobilenet_v2"` (ImageNet-pretrained encoder, ~1.6M
> params) and `LOSS_TYPE = "bayesian"` (Bayesian Loss+ on head points) are the defaults; set
> `"scratch"` / `"density"` to reproduce the original configuration for the ablation table. Each
> configuration gets its own Drive folder `MyDrive/RMSC-Net/runs/<encoder>_<loss>/`.

> **Runtime:** GPU (T4 or A100 recommended). Set *Runtime → Change runtime type → GPU* before running.


## 0  Runtime & GPU Check

In [ ]:
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout, result.stderr)

try:
    import torch
    print(f"PyTorch {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("[WARNING] No GPU detected — switch to a GPU runtime for training.")
except ImportError:
    print("PyTorch not yet installed — will be installed in the next cell.")


## 1  Install Dependencies

In [ ]:
%%capture
import subprocess, sys

packages = [
    "torch>=2.0.0",
    "torchvision>=0.15.0",
    "numpy>=1.24.0",
    "scipy>=1.10.0",
    "Pillow>=9.5.0",
    "opencv-python-headless>=4.7.0",
    "matplotlib>=3.7.0",
    "seaborn>=0.12.0",
    "tqdm>=4.65.0",
    "PyYAML>=6.0",
    "tensorboard>=2.13.0",
    "ptflops>=0.7.2",
    "h5py>=3.8.0",
    "scikit-learn>=1.2.0",
    "scikit-image>=0.20.0",
    "imgaug>=0.4.0",
    "albumentations>=1.3.0",
    "kornia>=0.6.12",
    "kagglehub>=0.2.0",
    "onnx", "onnxscript", "onnxruntime",  # ONNX export + verification (Section 24)
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages installed successfully.")


## 2  Mount Google Drive

Checkpoints, TensorBoard logs, metrics (CSV/JSON) and figures are all written to Drive, so they survive Colab timeouts. `save_json` / `save_csv` / `save_fig` / `update_summary` below are used by every evaluation cell; `results/summary.json` collects the headline numbers.


In [ ]:
from google.colab import drive
import os, json, datetime

DRIVE_ROOT = "/content/drive"
if not os.path.isdir(os.path.join(DRIVE_ROOT, "MyDrive")):
    drive.mount(DRIVE_ROOT)

# ---- experiment configuration --------------------------------------------------
ENCODER   = "mobilenet_v2"  # "mobilenet_v2" = ImageNet-pretrained encoder (default, much more
                            #   accurate); "scratch" = the original randomly-initialised encoder
LOSS_TYPE = "bayesian"      # "bayesian" = Bayesian Loss+ on head points (default);
                            #   "density"  = the original MSE-on-density-map loss
RUN_NAME  = f"{ENCODER}_{LOSS_TYPE}"  # one Drive folder per configuration, so runs never mix
                                      # (change it to start a fresh run of the same config)

# ---- personalise this path ----
PROJECT_DRIVE = os.path.join(DRIVE_ROOT, "MyDrive", "RMSC-Net")   # kaggle.json / dataset zip live here
RUN_DIR    = os.path.join(PROJECT_DRIVE, "runs", RUN_NAME)
CKPT_DIR   = os.path.join(RUN_DIR, "checkpoints")
LOG_DIR    = os.path.join(RUN_DIR, "logs")
RESULT_DIR = os.path.join(RUN_DIR, "results")
FIG_DIR    = os.path.join(RESULT_DIR, "figures")
EXPORT_DIR = os.path.join(RUN_DIR, "exported_models")

for d in [PROJECT_DRIVE, RUN_DIR, CKPT_DIR, LOG_DIR, RESULT_DIR, FIG_DIR, EXPORT_DIR]:
    os.makedirs(d, exist_ok=True)

SUMMARY_PATH = os.path.join(RESULT_DIR, "summary.json")


def _now():
    return datetime.datetime.now().isoformat(timespec="seconds")


def save_json(obj, name):
    path = os.path.join(RESULT_DIR, name)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=float)
    print(f"[Drive] saved {path}")
    return path


def save_csv(df, name, verbose=True, **kwargs):
    path = os.path.join(RESULT_DIR, name)
    df.to_csv(path, **kwargs)
    if verbose:
        print(f"[Drive] saved {path}")
    return path


def save_fig(fig, name, dpi=150):
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"[Drive] saved {path}")
    return path


def update_summary(section, values):
    """Merge `values` into results/summary.json under `section` (keeps earlier sections)."""
    summary = {}
    if os.path.exists(SUMMARY_PATH):
        with open(SUMMARY_PATH) as f:
            summary = json.load(f)
    summary[section] = dict(values, updated=_now())
    with open(SUMMARY_PATH, "w") as f:
        json.dump(summary, f, indent=2, default=float)
    print(f"[Drive] summary.json <- {section}")


print(f"Run          : {RUN_NAME}  (encoder={ENCODER}, loss={LOSS_TYPE})")
print(f"Drive root   : {RUN_DIR}")
print(f"Checkpoints  : {CKPT_DIR}")
print(f"Logs         : {LOG_DIR}")
print(f"Results      : {RESULT_DIR}  (figures in {FIG_DIR})")


## 3  Install RMSC-Net Code

**Option A** — Clone from GitHub (once you push the repo):
```bash
!git clone https://github.com/<your-username>/RMSC-Net.git /content/RMSC-Net
```
**Option B** — Upload the project ZIP from Drive (used below if no GitHub repo yet).

The cell below handles both cases automatically.


In [ ]:
import os, zipfile, shutil

CODE_DIR = "/content/RMSC-Net"

# Option A: GitHub clone (uncomment & edit after pushing)
# !git clone https://github.com/<your-username>/RMSC-Net.git {CODE_DIR}

# Option B: Upload ZIP from Drive
ZIP_PATH = os.path.join(PROJECT_DRIVE, "RMSC-Net.zip")

if os.path.exists(ZIP_PATH) and not os.path.exists(CODE_DIR):
    print("Extracting RMSC-Net.zip from Drive...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall("/content/")
    print("Done.")
elif os.path.exists(CODE_DIR):
    print(f"Code directory already exists at {CODE_DIR}")
else:
    # Create a minimal stub so later cells can import modules inline
    os.makedirs(CODE_DIR, exist_ok=True)
    # This is the normal case: the full model is defined inline in Section 4, so no external
    # code is needed. A GitHub clone / Drive ZIP is only an optional alternative.
    print("[OK] Using the inline RMSC-Net definitions in Section 4 (no external code needed).")

# Add to Python path
import sys
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)


## 4  Inline Model Definitions

Complete self-contained RMSC-Net implementation — no external files required.
These definitions replicate the full 27-file project in a single importable block.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Optional, Tuple

# ──────────────────────────────────────────────────────────────────────────────
# 4.1  Depthwise-Separable Conv helpers
# ──────────────────────────────────────────────────────────────────────────────
class DWConv(nn.Module):
    """Depthwise conv + BN + ReLU."""
    def __init__(self, c, k=3, d=1, bias=False):
        super().__init__()
        p = d * (k - 1) // 2
        self.dw = nn.Conv2d(c, c, k, padding=p, dilation=d, groups=c, bias=bias)
        self.bn = nn.BatchNorm2d(c)
    def forward(self, x):
        return F.relu(self.bn(self.dw(x)), inplace=True)

class DSConv(nn.Module):
    """Depthwise-separable conv block: DWConv → PWConv → BN → ReLU."""
    def __init__(self, cin, cout, k=3, d=1, bias=False):
        super().__init__()
        p = d * (k - 1) // 2
        self.dw = nn.Conv2d(cin, cin, k, padding=p, dilation=d, groups=cin, bias=bias)
        self.pw = nn.Conv2d(cin, cout, 1, bias=bias)
        self.bn = nn.BatchNorm2d(cout)
    def forward(self, x):
        return F.relu(self.bn(self.pw(self.dw(x))), inplace=True)

# ──────────────────────────────────────────────────────────────────────────────
# 4.2  Robust Feature Stem
# ──────────────────────────────────────────────────────────────────────────────
class RobustStem(nn.Module):
    """3×3 DWConv + 1×1 PWConv; stride-2 downsampling."""
    def __init__(self, in_ch=3, out_ch=32):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=2, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.ds    = DSConv(out_ch, out_ch)
    def forward(self, x):
        return self.ds(F.relu(self.bn1(self.conv1(x)), inplace=True))

# ──────────────────────────────────────────────────────────────────────────────
# 4.3  Lightweight Multi-Scale Encoder  (F1–F4)
# ──────────────────────────────────────────────────────────────────────────────
class EncoderBlock(nn.Module):
    def __init__(self, cin, cout, stride=2, n_ds=2):
        super().__init__()
        layers = [DSConv(cin, cout, k=3)]
        if stride == 2:
            layers.append(nn.MaxPool2d(2))
        for _ in range(n_ds - 1):
            layers.append(DSConv(cout, cout))
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        return self.block(x)

class LightweightEncoder(nn.Module):
    def __init__(self, stem_ch=32, channels=(64,128,256,256)):
        super().__init__()
        c = [stem_ch] + list(channels)
        self.b1 = EncoderBlock(c[0], c[1])
        self.b2 = EncoderBlock(c[1], c[2])
        self.b3 = EncoderBlock(c[2], c[3])
        self.b4 = EncoderBlock(c[3], c[4], stride=1)  # no spatial reduction at F4
    def forward(self, x):
        f1 = self.b1(x)
        f2 = self.b2(f1)
        f3 = self.b3(f2)
        f4 = self.b4(f3)
        return f1, f2, f3, f4

class PretrainedEncoder(nn.Module):
    """ImageNet-pretrained MobileNetV2 front-end producing F1-F4 with RMSC-Net's channel widths.

    F1 stride 4 (24 ch), F2 stride 8 (32 ch), F3 stride 16 (96 ch), F4 stride 32 (160 ch), each
    projected by 1x1 conv to `channels` so MSRM / DAA / fusion / decoder are unchanged. With only
    ~270 training images, starting from ImageNet features instead of random weights is the single
    largest accuracy gain available to a lightweight counter."""
    def __init__(self, channels=(64, 128, 256, 256), pretrained=True):
        super().__init__()
        import torchvision.models as tvm
        weights = tvm.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
        mb = tvm.mobilenet_v2(weights=weights).features
        self.stages = nn.ModuleList([mb[:4], mb[4:7], mb[7:14], mb[14:15]])
        self.proj = nn.ModuleList([
            nn.Sequential(nn.Conv2d(cin, cout, 1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
            for cin, cout in zip((24, 32, 96, 160), channels)
        ])
    def forward(self, x):
        feats = []
        for stage, proj in zip(self.stages, self.proj):
            x = stage(x)
            feats.append(proj(x))
        return tuple(feats)

# ──────────────────────────────────────────────────────────────────────────────
# 4.4  Multi-Scale Receptive-field Module (MSRM)
# ──────────────────────────────────────────────────────────────────────────────
class MSRMBranch(nn.Module):
    def __init__(self, ch, d):
        super().__init__()
        self.dw = DWConv(ch, d=d)
        self.pw = nn.Conv2d(ch, ch, 1, bias=False)
        self.bn = nn.BatchNorm2d(ch)
    def forward(self, x):
        return F.relu(self.bn(self.pw(self.dw(x))), inplace=True)

class MSRM(nn.Module):
    """Multi-Scale Receptive-field Module with dilated DWConv branches."""
    def __init__(self, ch=256, dilations=(1,2,3,5)):
        super().__init__()
        self.branches = nn.ModuleList([MSRMBranch(ch, d) for d in dilations])
        self.fuse = nn.Sequential(
            nn.Conv2d(ch * len(dilations), ch, 1, bias=False),
            nn.BatchNorm2d(ch),
            nn.ReLU(inplace=True),
        )
        self.res_proj = nn.Conv2d(ch, ch, 1, bias=False)
    def forward(self, x):
        outs = [b(x) for b in self.branches]
        fused = self.fuse(torch.cat(outs, dim=1))
        return fused + self.res_proj(x)

# ──────────────────────────────────────────────────────────────────────────────
# 4.5  Degradation-Aware Attention (DAA)
# ──────────────────────────────────────────────────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.gmp = nn.AdaptiveMaxPool2d(1)
        self.fc  = nn.Sequential(
            nn.Conv2d(ch, ch // r, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch // r, ch, 1, bias=False),
        )
    def forward(self, x):
        return torch.sigmoid(self.fc(self.gap(x)) + self.fc(self.gmp(x)))

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, k, padding=k // 2, bias=False)
    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        return torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))

class ReliabilityGate(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.gate = nn.Sequential(
            nn.Conv2d(ch, ch // 4, 1), nn.ReLU(inplace=True),
            nn.Conv2d(ch // 4, ch, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        return self.gate(self.gap(x)) * x

class DAA(nn.Module):
    """Degradation-Aware Attention: Channel → Spatial → Reliability Gate."""
    def __init__(self, ch=256, reduction=16):
        super().__init__()
        self.ca   = ChannelAttention(ch, reduction)
        self.sa   = SpatialAttention()
        self.gate = ReliabilityGate(ch)
    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return self.gate(x)

# ──────────────────────────────────────────────────────────────────────────────
# 4.6  Robust Multi-Scale Fusion
# ──────────────────────────────────────────────────────────────────────────────
class RobustFusion(nn.Module):
    def __init__(self, channels=(64,128,256,256), out_ch=256, mode="adaptive", target_index=1):
        super().__init__()
        self.mode = mode
        # Fuse at the resolution of feats[target_index]: F2 = 1/8 of the input, the resolution of
        # the ground-truth density maps. (Fusing at F1 = 1/4 and then upsampling x8 in the decoder
        # produced 2x-input-resolution maps that could not be compared with the 1/8 targets.)
        self.target_index = target_index
        self.align = nn.ModuleList([
            nn.Conv2d(c, out_ch, 1, bias=False) for c in channels
        ])
        if mode == "adaptive":
            self.weight_net = nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.Linear(out_ch * len(channels), len(channels)),
            )
        self.out_ch = out_ch
        self.n = len(channels)
    def forward(self, feats):
        target_h, target_w = feats[self.target_index].shape[2:]
        aligned = []
        for i, (f, proj) in enumerate(zip(feats, self.align)):
            a = proj(f)
            if a.shape[2] > target_h or a.shape[3] > target_w:
                a = F.adaptive_avg_pool2d(a, (target_h, target_w))   # finer map: pool down
            elif a.shape[2:] != (target_h, target_w):
                a = F.interpolate(a, size=(target_h, target_w),     # coarser map: upsample
                                  mode="bilinear", align_corners=False)
            aligned.append(a)
        if self.mode == "adaptive":
            cat = torch.cat(aligned, dim=1)
            w = torch.softmax(
                self.weight_net(cat).unsqueeze(-1).unsqueeze(-1),
                dim=1)
            # w: (B, n, 1, 1); w[:, i:i+1] keeps the channel axis so it broadcasts over
            # (B, C, H, W) for any batch size (w[:, i] only worked with batch size 1)
            out = sum(w[:, i:i+1] * aligned[i] for i in range(self.n))
        else:
            out = sum(aligned) / self.n
        return out

# ──────────────────────────────────────────────────────────────────────────────
# 4.7  Lightweight Density Decoder
# ──────────────────────────────────────────────────────────────────────────────
class DensityDecoder(nn.Module):
    """n_upsample DS-conv stages halving the channels (256->128->64->32), then a 1x1 conv.
    upsample=False keeps the fused 1/8 resolution, so the output matches the 1/8 GT density map."""
    def __init__(self, in_ch=256, n_upsample=3, upsample=False):
        super().__init__()
        layers = []
        c = in_ch
        for i in range(n_upsample):
            out_c = max(c // 2, 16)
            if upsample:
                layers.append(nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False))
            layers.append(DSConv(c, out_c))
            c = out_c
        layers.append(nn.Conv2d(c, 1, 1))
        self.decoder = nn.Sequential(*layers)
    def forward(self, x):
        return self.decoder(x)

# ──────────────────────────────────────────────────────────────────────────────
# 4.8  Full RMSC-Net
# ──────────────────────────────────────────────────────────────────────────────
class RMSCNet(nn.Module):
    """
    Input (B,3,H,W)
    → RobustStem → LightweightEncoder → MSRM (on F3) → DAA (on F3)
    → RobustFusion → DensityDecoder → density map (B,1,H/8,W/8)
    """
    def __init__(
        self,
        stem_ch: int = 32,
        enc_ch: Tuple[int,...] = (64, 128, 256, 256),
        msrm_ch: int = 256,
        dilations: Tuple[int,...] = (1, 2, 3, 5),
        daa_reduction: int = 16,
        fusion_ch: int = 256,
        fusion_mode: str = "adaptive",
        n_upsample: int = 3,
        encoder: str = "scratch",
        pretrained: bool = True,
    ):
        super().__init__()
        if encoder == "mobilenet_v2":
            self.stem    = nn.Identity()      # MobileNetV2 has its own stem
            self.encoder = PretrainedEncoder(enc_ch, pretrained)
        elif encoder == "scratch":
            self.stem    = RobustStem(3, stem_ch)
            self.encoder = LightweightEncoder(stem_ch, enc_ch)
        else:
            raise ValueError(f"unknown encoder {encoder!r}")
        self.msrm    = MSRM(msrm_ch, dilations)
        self.daa     = DAA(msrm_ch, daa_reduction)
        self.fusion  = RobustFusion(enc_ch, fusion_ch, fusion_mode)
        self.decoder = DensityDecoder(fusion_ch, n_upsample)

    def forward(self, x):
        s       = self.stem(x)
        f1,f2,f3,f4 = self.encoder(s)
        f3m     = self.msrm(f3)
        f3d     = self.daa(f3m)
        fused   = self.fusion([f1, f2, f3d, f4])
        density = self.decoder(fused)
        return F.relu(density, inplace=True)

    def backbone_parameters(self):
        """Pretrained parameters (fine-tuned at a lower LR); empty for the scratch encoder."""
        if isinstance(self.encoder, PretrainedEncoder):
            return list(self.encoder.stages.parameters())
        return []


MODEL_KWARGS = dict(encoder=ENCODER)   # every RMSCNet(...) in the notebook uses this

print("RMSC-Net model defined successfully.")


## 5  Model Sanity Check & Parameter Count

In [ ]:
import torch
from ptflops import get_model_complexity_info

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

model = RMSCNet(**MODEL_KWARGS).to(DEVICE)
model.eval()

# Forward pass
dummy = torch.randn(1, 3, 512, 512).to(DEVICE)
model_summary = {}
with torch.no_grad():
    out = model(dummy)
print(f"Input  shape : {tuple(dummy.shape)}")
print(f"Output shape : {tuple(out.shape)}")
assert out.shape[-2:] == (512 // 8, 512 // 8), "density map must be 1/8 of the input"

# Count params
total   = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params     : {total:,}")
print(f"Trainable params : {trainable:,}  ({trainable/1e6:.2f} M)")

# FLOPs
try:
    macs, params = get_model_complexity_info(
        model, (3, 512, 512), as_strings=True,
        print_per_layer_stat=False, verbose=False
    )
    print(f"MACs (FLOPs/2)   : {macs}")
    print(f"Params (ptflops) : {params}")
    model_summary["MACs_512x512"] = macs
except Exception as e:
    print(f"ptflops estimate skipped ({e})")

# FPS
import time
model.eval()
n_runs = 50
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(n_runs):
        _ = model(dummy)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
fps = n_runs / (time.perf_counter() - t0)
print(f"Inference FPS    : {fps:.1f}")

model_summary.update({
    "total_params": total, "trainable_params": trainable,
    "input_shape": list(dummy.shape), "output_shape": list(out.shape),
    "fps_512x512": round(fps, 2),
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
})
save_json(model_summary, "model_summary.json")
update_summary("model", model_summary)


## 6  ShanghaiTech Part A Dataset (downloaded automatically)

The cell below gets the dataset by itself, trying in order:

1. an already-downloaded copy in this runtime (re-running the cell is instant);
2. **Kaggle `tthien/shanghaitech` via `kagglehub`**: no manual step needed inside Colab. If Kaggle
   asks for credentials, put your `kaggle.json` (Kaggle → *Settings* → *Create New Token*) at
   `MyDrive/RMSC-Net/kaggle.json` and it is picked up automatically;
3. a `ShanghaiTech.zip` you placed at `MyDrive/RMSC-Net/ShanghaiTech.zip` (offline fallback).

It then verifies the expected layout (300 train / 182 test images, one `GT_IMG_*.mat` per image):
```
part_A/
  train_data/  images/ (300 JPGs)   ground-truth/ (300 MAT files)
  test_data/   images/ (182 JPGs)   ground-truth/ (182 MAT files)
```


In [ ]:
import os, zipfile, glob, shutil

DATA_ZIP  = os.path.join(PROJECT_DRIVE, "ShanghaiTech.zip")
DATA_ROOT = "/content/ShanghaiTech"
KAGGLE_DATASET = "tthien/shanghaitech"


def find_part_a(root):
    """Return the first part_A dir under root that has images AND .mat ground truth."""
    for dirpath, _, _ in os.walk(root):
        if os.path.basename(dirpath) != "part_A":
            continue
        if (glob.glob(os.path.join(dirpath, "train_data", "images", "*.jpg"))
                and glob.glob(os.path.join(dirpath, "train_data", "ground-truth", "*.mat"))):
            return dirpath
    return None


PART_A = None

# 1) already available in this runtime
for root in [DATA_ROOT, "/kaggle/input"]:
    if os.path.isdir(root):
        PART_A = find_part_a(root)
        if PART_A:
            print(f"Dataset already available: {PART_A}")
            break

# 2) Kaggle via kagglehub (uses Drive's kaggle.json if present)
if PART_A is None:
    kaggle_json = os.path.join(PROJECT_DRIVE, "kaggle.json")
    if os.path.exists(kaggle_json):
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        shutil.copy(kaggle_json, os.path.expanduser("~/.kaggle/kaggle.json"))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
        print("Using kaggle.json from Drive.")
    try:
        import kagglehub
        print(f"Downloading {KAGGLE_DATASET} from Kaggle ...")
        dl_path = kagglehub.dataset_download(KAGGLE_DATASET)
        PART_A = find_part_a(dl_path)
        print(f"Downloaded to {dl_path}")
    except Exception as e:
        print(f"[WARNING] Kaggle download failed: {e}")

# 3) zip on Drive
if PART_A is None and os.path.exists(DATA_ZIP):
    print("Unzipping ShanghaiTech.zip from Drive ...")
    with zipfile.ZipFile(DATA_ZIP, "r") as zf:
        zf.extractall(DATA_ROOT)
    PART_A = find_part_a(DATA_ROOT)

if PART_A is None:
    raise RuntimeError(
        "Could not obtain ShanghaiTech Part A. Either put kaggle.json at "
        f"{os.path.join(PROJECT_DRIVE, 'kaggle.json')} (Kaggle -> Settings -> Create New Token) or "
        f"upload the dataset zip to {DATA_ZIP}, then re-run this cell.")

# Verify
train_imgs = sorted(glob.glob(os.path.join(PART_A, "train_data", "images", "*.jpg")))
test_imgs  = sorted(glob.glob(os.path.join(PART_A, "test_data",  "images", "*.jpg")))
missing_gt = [p for p in train_imgs + test_imgs
              if not os.path.exists(p.replace("images", "ground-truth")
                                      .replace(".jpg", ".mat").replace("IMG_", "GT_IMG_"))]
print(f"Part A root  : {PART_A}")
print(f"Train images : {len(train_imgs)}  |  Test images : {len(test_imgs)}  |  missing GT : {len(missing_gt)}")
assert len(train_imgs) == 300 and len(test_imgs) == 182 and not missing_gt, "unexpected dataset layout"

# Official split: last 30 of 300 training images used as validation
VAL_IMGS   = train_imgs[-30:]
TRAIN_IMGS = train_imgs[:-30]
print(f"→  Training  : {len(TRAIN_IMGS)}  |  Validation : {len(VAL_IMGS)}  |  Test : {len(test_imgs)}")
save_json({"part_A_root": PART_A, "train": [os.path.basename(p) for p in TRAIN_IMGS],
           "val": [os.path.basename(p) for p in VAL_IMGS],
           "test": [os.path.basename(p) for p in test_imgs]}, "data_split.json")


## 7  Dataset & DataLoader

In [ ]:
import numpy as np
import scipy.io as sio
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import random, math

DENSITY_SCALE = 8   # GT density map is at 1/8 input resolution and SUMS TO THE HEAD COUNT
PAD_MULTIPLE  = 32  # the encoder downsamples by up to 32; pad images to a multiple of it so the
                    # network output and the 1/8 GT map always have identical sizes
FILL_RGB = tuple(int(round(m * 255)) for m in (0.485, 0.456, 0.406))  # pads become 0 after Normalize


def pad_image(img, min_h=0, min_w=0, multiple=PAD_MULTIPLE):
    """Pad a PIL image at the bottom/right (head coordinates are unchanged)."""
    w, h = img.size
    H = max(h, min_h); W = max(w, min_w)
    H = int(math.ceil(H / multiple) * multiple); W = int(math.ceil(W / multiple) * multiple)
    if (H, W) == (h, w):
        return img
    canvas = Image.new("RGB", (W, H), FILL_RGB)
    canvas.paste(img, (0, 0))
    return canvas

class ShanghaiTechDataset(Dataset):
    """ShanghaiTech Part A dataset with optional on-the-fly augmentation."""

    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]

    def __init__(self, img_paths, mode="train", crop_size=256, crops_per_image=1, scale_range=None):
        self.paths     = img_paths
        self.mode      = mode
        self.crop_size = crop_size
        # several random crops per image per epoch (train only): ~4x more gradient steps per epoch
        self.crops_per_image = crops_per_image if mode == "train" else 1
        self.scale_range = scale_range if mode == "train" else None
        self.norm      = T.Normalize(self.MEAN, self.STD)

    def _load_density(self, img_path):
        gt_path = img_path.replace("images", "ground-truth") \
                           .replace(".jpg", ".mat")
        gt_path = gt_path.replace("IMG_", "GT_IMG_")
        mat = sio.loadmat(gt_path)
        # ShanghaiTech stores the density map in field 'image_info'
        try:
            points = mat["image_info"][0, 0][0, 0][0]
        except Exception:
            points = mat.get("annPoints", np.zeros((0, 2)))
        return points

    def _make_density_map(self, points, h, w):
        """Gaussian kernel density map at 1/DENSITY_SCALE resolution."""
        dm_h, dm_w = h // DENSITY_SCALE, w // DENSITY_SCALE
        density = np.zeros((dm_h, dm_w), dtype=np.float32)
        if len(points) == 0:
            return density
        from scipy.ndimage import gaussian_filter
        for pt in points:
            px = min(int(pt[0] / DENSITY_SCALE), dm_w - 1)
            py = min(int(pt[1] / DENSITY_SCALE), dm_h - 1)
            density[py, px] += 1.0
        density = gaussian_filter(density, sigma=3)
        # Scale so sum gives correct count
        count = len(points)
        s = density.sum()
        if s > 0:
            density = density * count / s
        return density

    def __len__(self):
        return len(self.paths) * self.crops_per_image

    def __getitem__(self, idx):
        idx = idx % len(self.paths)
        img = Image.open(self.paths[idx]).convert("RGB")
        points = np.asarray(self._load_density(self.paths[idx]), dtype=np.float32).reshape(-1, 2)
        w0, h0 = img.size
        points = points[(points[:, 0] >= 0) & (points[:, 0] < w0) & (points[:, 1] >= 0) & (points[:, 1] < h0)]
        cs = self.crop_size
        if self.scale_range is not None:
            # random rescale (head sizes in Part A vary enormously); points scale with the image
            s = random.uniform(*self.scale_range)
            nw, nh = max(1, int(round(w0 * s))), max(1, int(round(h0 * s)))
            img = img.resize((nw, nh), Image.BILINEAR)
            points = points * np.array([nw / w0, nh / h0], dtype=np.float32)
        # Pad (never resize): training crops need every image to be at least crop_size, and all
        # images need sizes that are multiples of 16 (see PAD_MULTIPLE).
        if self.mode == "train":
            img = pad_image(img, cs, cs)
        else:
            img = pad_image(img)
        w, h = img.size
        density = self._make_density_map(points, h, w)

        # Augment
        if self.mode == "train":
            # Random crop, aligned to the 8-px density grid so image and density stay registered,
            # and of fixed size so a batch can be stacked
            top  = random.randrange(0, h - cs + 1, DENSITY_SCALE)
            left = random.randrange(0, w - cs + 1, DENSITY_SCALE)
            img  = TF.crop(img, top, left, cs, cs)
            dm_top, dm_left, dm_cs = top // DENSITY_SCALE, left // DENSITY_SCALE, cs // DENSITY_SCALE
            density = density[dm_top:dm_top+dm_cs, dm_left:dm_left+dm_cs]
            points = points - np.array([left, top], dtype=np.float32)
            points = points[(points[:, 0] >= 0) & (points[:, 0] < cs) & (points[:, 1] >= 0) & (points[:, 1] < cs)]
            # Random flip
            if random.random() > 0.5:
                img     = TF.hflip(img)
                density = np.fliplr(density).copy()
                points  = points.copy()
                points[:, 0] = cs - points[:, 0]
            count = float(len(points))  # heads inside THIS crop, not the whole image
        else:
            count = float(points.shape[0])

        img_t = self.norm(TF.to_tensor(img))
        den_t = torch.from_numpy(np.ascontiguousarray(density)).unsqueeze(0)
        if self.mode == "train":  # training also returns the head points (for the Bayesian loss)
            return (img_t, den_t, torch.tensor(count, dtype=torch.float32),
                    torch.from_numpy(np.ascontiguousarray(points, dtype=np.float32)))
        return img_t, den_t, torch.tensor(count, dtype=torch.float32)


def collate_train(batch):
    # images/densities/counts stack as usual; point sets differ in length, so they stay a list
    imgs, dens, counts, points = zip(*batch)
    return torch.stack(imgs), torch.stack(dens), torch.stack(counts), list(points)


# Build loaders (only if dataset was extracted)
if PART_A and os.path.exists(PART_A):
    BATCH_SIZE      = 8
    CROP_SIZE       = 256          # smaller crops + several per image -> many more updates
    CROPS_PER_IMAGE = 4
    SCALE_RANGE     = (0.7, 1.3)   # random rescale augmentation
    NUM_WORKERS     = 2

    train_ds = ShanghaiTechDataset(TRAIN_IMGS, "train", CROP_SIZE, CROPS_PER_IMAGE, SCALE_RANGE)
    val_ds   = ShanghaiTechDataset(VAL_IMGS,   "val",   CROP_SIZE)
    test_ds  = ShanghaiTechDataset(test_imgs,  "val",   CROP_SIZE)

    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                              pin_memory=True, drop_last=True, collate_fn=collate_train)
    val_loader   = DataLoader(val_ds,   1, shuffle=False, num_workers=1)
    test_loader  = DataLoader(test_ds,  1, shuffle=False, num_workers=1)

    print(f"Train batches  : {len(train_loader)}")
    print(f"Val   batches  : {len(val_loader)}")
    print(f"Test  batches  : {len(test_loader)}")
else:
    print("[SKIP] Dataset not found — loaders not created.")


## 8  Composite Loss Function

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DENSITY_AMPLIFY = 100.0  # density maps hold tiny per-pixel values (heads per 8x8 cell), so
                         # plain MSE on them gives almost no gradient; MSE terms compare maps x100


class BayesianLoss(nn.Module):
    """Bayesian Loss+ (Ma et al., ICCV 2019): supervision from the annotated head POINTS.
    Each output pixel is softly assigned to the heads (and a background model) with a Gaussian
    likelihood; the expected count each head receives must be 1 (background: 0). Avoids the
    hand-picked Gaussian blur of density-map targets."""
    def __init__(self, stride=8, sigma=8.0, bg_dist=512.0, use_bg=True):
        super().__init__()
        self.stride, self.sigma, self.bg_dist, self.use_bg = stride, sigma, bg_dist, use_bg

    def forward(self, pred, points_list):
        _, _, h, w = pred.shape
        ys = (torch.arange(h, device=pred.device, dtype=torch.float32) + 0.5) * self.stride
        xs = (torch.arange(w, device=pred.device, dtype=torch.float32) + 0.5) * self.stride
        losses = []
        for b, pts in enumerate(points_list):
            dens = pred[b, 0].reshape(-1)
            pts = pts.to(pred.device, torch.float32)
            if pts.numel() == 0:
                losses.append(dens.sum().abs())
                continue
            dist = ((pts[:, 1:2] - ys[None]) ** 2)[:, :, None] + ((pts[:, 0:1] - xs[None]) ** 2)[:, None, :]
            dist = dist.reshape(pts.size(0), -1)
            if self.use_bg:
                bg = self.bg_dist ** 2 / (dist.min(dim=0, keepdim=True)[0].clamp_min(0) + 1e-5)
                dist = torch.cat([dist, bg], dim=0)
            post = torch.softmax(-dist / (2 * self.sigma ** 2), dim=0)
            expected = (post * dens[None]).sum(dim=1)
            target = torch.ones_like(expected)
            if self.use_bg:
                target[-1] = 0.0
            losses.append((target - expected).abs().sum())
        return torch.stack(losses).mean()


class RMSCNetLoss(nn.Module):
    """
    L = λ_d·L_main + λ_c·L_count + λ_cons·L_consistency + λ_f·L_feature
    L_main = Bayesian loss on head points (LOSS_TYPE="bayesian") or density-map MSE ("density")
    """
    def __init__(self, lam_d=1.0, lam_c=0.1, lam_cons=0.5, lam_f=0.1, loss_type="density"):
        super().__init__()
        self.lam_d, self.lam_c = lam_d, lam_c
        self.lam_cons, self.lam_f = lam_cons, lam_f
        self.bayes = BayesianLoss(stride=DENSITY_SCALE) if loss_type == "bayesian" else None

    def density_loss(self, pred, gt):
        return F.mse_loss(pred * DENSITY_AMPLIFY, gt * DENSITY_AMPLIFY)

    def count_loss(self, pred, gt_count):
        pred_count = pred.sum(dim=(1,2,3))  # density maps sum to the count
        return F.l1_loss(pred_count, gt_count)

    def consistency_loss(self, pred_clean, pred_corrupt):
        return F.mse_loss(pred_clean * DENSITY_AMPLIFY, pred_corrupt * DENSITY_AMPLIFY)

    def feature_loss(self, feat_clean, feat_corrupt):
        return F.cosine_embedding_loss(
            feat_clean.flatten(1),
            feat_corrupt.flatten(1),
            torch.ones(feat_clean.shape[0], device=feat_clean.device),
        )

    def forward(self, pred, gt_density, gt_count,
                pred_corrupt=None, pred_clean_stop=None,
                feat_clean=None, feat_corrupt=None, points=None):
        if self.bayes is not None and points is not None:
            L = self.lam_d * self.bayes(pred, points)
        else:
            L = self.lam_d * self.density_loss(pred, gt_density)
        L = L + self.lam_c * self.count_loss(pred, gt_count)
        if pred_corrupt is not None and pred_clean_stop is not None:
            L = L + self.lam_cons * self.consistency_loss(
                pred_clean_stop.detach(), pred_corrupt)
        if feat_clean is not None and feat_corrupt is not None:
            L = L + self.lam_f * self.feature_loss(feat_clean, feat_corrupt)
        return L

criterion = RMSCNetLoss(lam_d=1.0, lam_c=0.1, lam_cons=0.5, lam_f=0.1, loss_type=LOSS_TYPE)
print(f"Loss function ready ({LOSS_TYPE}).")


## 9  Stage I — Clean Curriculum Training (100 epochs)

Trains on clean images only with density + count loss.

**Checkpointing & early stopping (all three stages):**
- After **every epoch** the full stage state (weights, optimizer, LR scheduler, history, RNG) is
  written atomically to `MyDrive/RMSC-Net/checkpoints/stageN_last.pth`. After a Colab timeout,
  reconnect and **Runtime → Run all**: finished stages are skipped, an interrupted stage resumes
  from its next epoch, at most one epoch is lost.
- `stageN_best.pth` is updated whenever val MAE improves; evaluation always uses the best one.
- **Early stopping:** a stage ends once its smoothed (3-epoch average) val MAE hasn't improved for
  `S1/S2/S3_PATIENCE` epochs (20 / 20 / 10), never before `S*_MIN_EPOCHS` (30 / 20 / 10). The
  epoch counts (100 / 100 / 50) are upper limits. Set `EARLY_STOP = False` to always run them all.


In [ ]:
import os, time, math, json, random
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm

# ── Hyper-parameters ──────────────────────────────────────────────────────────
LR            = 1e-4  # new layers; the pretrained encoder uses LR * BACKBONE_LR_MULT
BACKBONE_LR_MULT = 0.5
WEIGHT_DECAY  = 1e-4
WARMUP_EPOCHS = 5
GRAD_CLIP     = 5.0
STAGE1_EPOCHS = 300   # upper limit; early stopping ends it once val MAE stops improving
FORCE_RETRAIN = False # True = retrain stages that already finished (their Drive results are kept
                      # until overwritten by a better checkpoint)

# ── Early stopping (all stages) ──────────────────────────────────────────────
EARLY_STOP     = True # stop a stage once its smoothed val MAE stops improving
ES_SMOOTHING   = 5    # moving average over this many epochs (only 30 val images -> noisy MAE)
ES_MIN_DELTA   = 0.0  # an improvement must beat the best by more than this
S1_PATIENCE    = 40   # epochs without improvement before Stage I stops ...
S1_MIN_EPOCHS  = 60   # ... but never before this epoch


# ── Stage bookkeeping on Drive (shared by Stages I-III) ─────────────────────
def stage_done_path(stage):
    return os.path.join(CKPT_DIR, f"stage{stage}_done.json")


def stage_already_done(stage):
    if FORCE_RETRAIN or not os.path.exists(stage_done_path(stage)):
        return False
    with open(stage_done_path(stage)) as f:
        info = json.load(f)
    print(f"Stage {stage} already finished on Drive ({info}) -- skipping. "
          "Set FORCE_RETRAIN = True to train it again.")
    return True


def mark_stage_done(stage, info):
    with open(stage_done_path(stage), "w") as f:
        json.dump(dict(info, finished=_now()), f, indent=2, default=float)
    update_summary(f"stage{stage}", info)


def best_so_far(ckpt_path):
    # Never let a fresh (re)run overwrite a better checkpoint already on Drive with a worse one.
    if os.path.exists(ckpt_path):
        return float(torch.load(ckpt_path, map_location="cpu").get("val_mae", float("inf")))
    return float("inf")


def log_epoch(stage, row, history):
    history.append(row)
    save_csv(pd.DataFrame(history), f"stage{stage}_history.csv", verbose=False, index=False)


# ── Per-epoch checkpointing / resume (all stages) ──────────────────────────
def atomic_save(obj, path):
    # write to a temp file, then rename: a runtime that dies mid-write can't corrupt the checkpoint
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def stage_last_path(stage):
    return os.path.join(CKPT_DIR, f"stage{stage}_last.pth")


def save_stage_state(stage, epoch, model, optimizer, scheduler, best_mae, history):
    """Full resumable state of a stage, written to Drive after every epoch."""
    atomic_save({
        "stage": stage, "epoch": epoch,
        "state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_mae": best_mae, "history": history,
        "rng": {"python": random.getstate(), "numpy": np.random.get_state(),
                "torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None},
    }, stage_last_path(stage))


def load_stage_state(stage, model, optimizer, scheduler):
    """Resume an interrupted stage. Returns (next_epoch, best_mae, history) or None."""
    path = stage_last_path(stage)
    if FORCE_RETRAIN or not os.path.exists(path):
        return None
    try:
        # weights_only=False: our own file, which also holds history/RNG (Python/NumPy objects)
        state = torch.load(path, map_location=DEVICE, weights_only=False)
    except Exception as e:
        print(f"[WARNING] could not read {path} ({e}); restarting Stage {stage} from epoch 1.")
        return None
    model.load_state_dict(state["state_dict"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    rng = state["rng"]
    random.setstate(rng["python"]); np.random.set_state(rng["numpy"])
    torch.set_rng_state(rng["torch"].cpu())
    if rng["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in rng["cuda"]])
    print(f"Resuming Stage {stage} from Drive: {state['epoch']} epoch(s) done, "
          f"best val MAE {state['best_mae']:.2f}")
    return state["epoch"] + 1, state["best_mae"], state["history"]


def early_stop_check(history, patience, min_epochs):
    """Decided from the (checkpointed) history alone, so it behaves the same after a resume."""
    vals = [row["val_mae"] for row in history]
    if not EARLY_STOP or len(vals) < max(min_epochs, 1):
        return False, ""
    k = max(ES_SMOOTHING, 1)
    smoothed = [float(np.mean(vals[max(0, i - k + 1):i + 1])) for i in range(len(vals))]
    best_i, best = 0, smoothed[0]
    for i, v in enumerate(smoothed):
        if v < best - ES_MIN_DELTA:
            best_i, best = i, v
    since = len(smoothed) - 1 - best_i
    if since >= patience:
        return True, (f"smoothed val MAE not improved for {since} epochs "
                      f"(best {best:.2f} at epoch {history[best_i]['epoch']})")
    return False, ""

# ── Model / Optimiser ─────────────────────────────────────────────────────────
model     = RMSCNet(**MODEL_KWARGS).to(DEVICE)
criterion = RMSCNetLoss(loss_type=LOSS_TYPE)
_bb_ids   = {id(p) for p in model.backbone_parameters()}
optimizer = optim.Adam([
    {"params": [p for p in model.parameters() if id(p) not in _bb_ids], "lr": LR},
    {"params": model.backbone_parameters(), "lr": LR * BACKBONE_LR_MULT},
], weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=STAGE1_EPOCHS - WARMUP_EPOCHS, eta_min=LR * 0.01)

writer    = SummaryWriter(log_dir=os.path.join(LOG_DIR, "stage1"))
best_ckpt = os.path.join(CKPT_DIR, "stage1_best.pth")
best_mae  = best_so_far(best_ckpt)
history_s1 = []

def warmup_lr(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    return 1.0

def evaluate(loader, net, device):
    net.eval()
    maes, rmses = [], []
    with torch.no_grad():
        for imgs, dens, counts in loader:
            imgs   = imgs.to(device)
            pred   = net(imgs)
            p_cnt  = pred.sum().item()
            g_cnt  = counts.item()
            maes.append(abs(p_cnt - g_cnt))
            rmses.append((p_cnt - g_cnt) ** 2)
    mae  = float(np.mean(maes))
    rmse = float(np.sqrt(np.mean(rmses)))
    return mae, rmse

# ── Training loop ─────────────────────────────────────────────────────────────
if 'train_loader' in dir() and not stage_already_done(1):
    print(f"Starting Stage I — {STAGE1_EPOCHS} epochs")
    start_epoch, stopped_early, stop_reason = 1, False, ""
    resumed = load_stage_state(1, model, optimizer, scheduler)
    if resumed:
        start_epoch, best_mae, history_s1 = resumed
        stopped_early, stop_reason = early_stop_check(history_s1, S1_PATIENCE, S1_MIN_EPOCHS)
        if stopped_early:  # interrupted after the stopping epoch was saved
            start_epoch = STAGE1_EPOCHS + 1

    for epoch in range(start_epoch, STAGE1_EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        t0 = time.time()

        # Warmup
        if epoch <= WARMUP_EPOCHS:
            for pg in optimizer.param_groups:   # each group warms up to its own base LR
                pg["lr"] = pg["initial_lr"] * warmup_lr(epoch - 1)

        for imgs, dens, counts, points in tqdm(train_loader,
                                               desc=f"S1 Ep {epoch}/{STAGE1_EPOCHS}",
                                               leave=False):
            imgs, dens, counts = imgs.to(DEVICE), dens.to(DEVICE), counts.to(DEVICE)
            pred = model(imgs)
            loss = criterion(pred, dens, counts, points=points)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            epoch_loss += loss.item()

        if epoch > WARMUP_EPOCHS:
            scheduler.step()

        avg_loss = epoch_loss / len(train_loader)
        val_mae, val_rmse = evaluate(val_loader, model, DEVICE)

        writer.add_scalar("Loss/train", avg_loss, epoch)
        writer.add_scalar("Val/MAE",   val_mae,   epoch)
        writer.add_scalar("Val/RMSE",  val_rmse,  epoch)
        writer.add_scalar("LR",        optimizer.param_groups[0]["lr"], epoch)
        writer.flush()
        log_epoch(1, {"epoch": epoch, "train_loss": avg_loss, "val_mae": val_mae,
                      "val_rmse": val_rmse, "lr": optimizer.param_groups[0]["lr"]}, history_s1)

        if val_mae < best_mae:
            best_mae = val_mae
            atomic_save({"epoch": epoch, "state_dict": model.state_dict(),
                         "optimizer": optimizer.state_dict(),
                         "val_mae": val_mae}, best_ckpt)

        if epoch % 10 == 0 or epoch == STAGE1_EPOCHS:
            print(f"[S1 {epoch:3d}/{STAGE1_EPOCHS}] "
                  f"loss={avg_loss:.4f}  val_MAE={val_mae:.2f}  "
                  f"val_RMSE={val_rmse:.2f}  "
                  f"lr={optimizer.param_groups[0]['lr']:.6f}  "
                  f"time={time.time()-t0:.1f}s")

        # full state to Drive every epoch, then the early-stopping decision
        save_stage_state(1, epoch, model, optimizer, scheduler, best_mae, history_s1)
        stopped_early, stop_reason = early_stop_check(history_s1, S1_PATIENCE, S1_MIN_EPOCHS)
        if stopped_early:
            print(f"[S1] Early stopping at epoch {epoch}: {stop_reason}")
            break

    writer.close()
    mark_stage_done(1, {"epochs": STAGE1_EPOCHS, "epochs_run": len(history_s1),
                        "stopped_early": stopped_early, "stop_reason": stop_reason, "best_val_mae": best_mae, "ckpt": best_ckpt})
    print(f"Stage I done after {len(history_s1)} epochs. Best val MAE: {best_mae:.2f}  ckpt: {best_ckpt}")
elif 'train_loader' not in dir():
    print("[SKIP] train_loader not available — run Section 7 first.")


## 10  Environmental Corruption Pipeline

Implements 12 corruption types × 5 severity levels used in Stage II training
and full robustness evaluation.


In [ ]:
import cv2
import numpy as np
from PIL import Image
import torch
import torchvision.transforms.functional as TF
import torchvision.transforms as T
import random, math

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def denorm(t):
    """Tensor (3,H,W) normalised → uint8 numpy (H,W,3)."""
    x = (t.cpu() * STD + MEAN).clamp(0, 1).numpy()
    return (np.transpose(x, (1, 2, 0)) * 255).astype(np.uint8)

def renorm(arr):
    """uint8 numpy (H,W,3) → normalised tensor (3,H,W)."""
    norm = T.Normalize(MEAN.squeeze().tolist(), STD.squeeze().tolist())
    return norm(TF.to_tensor(Image.fromarray(arr)))


SEVERITY_PARAMS = {
    "gaussian_noise":  [0.04, 0.06, 0.08, 0.12, 0.18],
    "shot_noise":      [60, 25, 12, 5, 3],
    "impulse_noise":   [0.03, 0.06, 0.09, 0.17, 0.27],
    "defocus_blur":    [1, 2, 3, 4, 6],
    "motion_blur":     [3, 5, 8, 12, 15],
    "fog":             [0.2, 0.3, 0.5, 0.7, 0.9],
    "frost":           [0.1, 0.15, 0.2, 0.3, 0.4],
    "snow":            [0.05, 0.1, 0.15, 0.2, 0.3],
    "rain":            [0.05, 0.1, 0.15, 0.2, 0.3],
    "brightness":      [0.1, 0.2, 0.35, 0.5, 0.65],
    "contrast":        [0.75, 0.5, 0.4, 0.3, 0.15],
    "jpeg_compression":[80, 60, 40, 25, 10],
}

def apply_corruption(img_np, corruption_type, severity):
    """Apply corruption to uint8 (H,W,3) numpy array. Returns same type."""
    p = SEVERITY_PARAMS[corruption_type][severity - 1]
    img = img_np.astype(np.float32) / 255.0

    if corruption_type == "gaussian_noise":
        img = (img + np.random.randn(*img.shape) * p).clip(0, 1)

    elif corruption_type == "shot_noise":
        img = np.random.poisson(img * p) / p
        img = img.clip(0, 1)

    elif corruption_type == "impulse_noise":
        mask = np.random.random(img.shape[:2]) < p
        # salt-and-pepper: one value per pixel, shape (N, 1) so it broadcasts over the 3 channels
        img[mask] = np.random.randint(0, 2, (mask.sum(), 1)).astype(np.float32)

    elif corruption_type == "defocus_blur":
        k = 2 * p + 1
        kk = np.zeros((k, k), np.float32)
        cv2.circle(kk, (p, p), p, 1, -1)
        kk /= kk.sum()
        img = np.stack([cv2.filter2D(img[:,:,c], -1, kk) for c in range(3)], -1)

    elif corruption_type == "motion_blur":
        k = p
        kernel = np.zeros((k, k), np.float32)
        kernel[k//2, :] = 1.0 / k
        img = np.stack([cv2.filter2D(img[:,:,c], -1, kernel) for c in range(3)], -1)

    elif corruption_type == "fog":
        img = img * (1 - p) + p

    elif corruption_type in ("frost", "snow", "rain"):
        noise = np.random.randn(*img.shape).astype(np.float32) * p
        img   = (img + noise).clip(0, 1)

    elif corruption_type == "brightness":
        img = (img + p).clip(0, 1)

    elif corruption_type == "contrast":
        mean  = img.mean(axis=(0, 1), keepdims=True)
        img   = (img - mean) * p + mean
        img   = img.clip(0, 1)

    elif corruption_type == "jpeg_compression":
        img_u8 = (img * 255).astype(np.uint8)
        encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), int(p)]
        _, enc = cv2.imencode(".jpg", cv2.cvtColor(img_u8, cv2.COLOR_RGB2BGR), encode_param)
        dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(dec, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

    return (img * 255).astype(np.uint8)


def corrupt_batch(imgs_tensor, corruption_type, severity):
    """Apply corruption to a normalised (B,3,H,W) tensor. Returns same shape tensor."""
    out = []
    for i in range(imgs_tensor.shape[0]):
        arr  = denorm(imgs_tensor[i])
        arr2 = apply_corruption(arr, corruption_type, severity)
        out.append(renorm(arr2))
    return torch.stack(out).to(imgs_tensor.device)

print(f"Corruption pipeline ready: {len(SEVERITY_PARAMS)} types × 5 severities")


## 11  Stage II — Consistency Training with Corruptions (100 epochs)

Loads Stage I weights and continues training with:
- Clean images (density + count loss)
- Corrupted images (density + count + consistency + feature loss)


In [ ]:
import os, time, random
import torch
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm

STAGE2_EPOCHS = 100  # upper limit (early stopping usually ends sooner)
LR_S2         = 5e-5
S2_PATIENCE   = 20
S2_MIN_EPOCHS = 20

# Load Stage I checkpoint
s1_ckpt = os.path.join(CKPT_DIR, "stage1_best.pth")
best_ckpt_s2 = os.path.join(CKPT_DIR, "stage2_best.pth")

if 'train_loader' in dir() and os.path.exists(s1_ckpt) and not stage_already_done(2):
    ckpt = torch.load(s1_ckpt, map_location=DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    print(f"Loaded Stage I checkpoint (val_MAE={ckpt['val_mae']:.2f})")

    optimizer_s2 = optim.Adam(model.parameters(), lr=LR_S2, weight_decay=WEIGHT_DECAY)
    scheduler_s2 = optim.lr_scheduler.CosineAnnealingLR(
        optimizer_s2, T_max=STAGE2_EPOCHS, eta_min=LR_S2 * 0.01)
    writer_s2 = SummaryWriter(log_dir=os.path.join(LOG_DIR, "stage2"))

    CORRUPT_TYPES = list(SEVERITY_PARAMS.keys())
    best_mae_s2   = best_so_far(best_ckpt_s2)
    history_s2    = []

    start_epoch, stopped_early, stop_reason = 1, False, ""
    resumed = load_stage_state(2, model, optimizer_s2, scheduler_s2)
    if resumed:
        start_epoch, best_mae_s2, history_s2 = resumed
        stopped_early, stop_reason = early_stop_check(history_s2, S2_PATIENCE, S2_MIN_EPOCHS)
        if stopped_early:  # interrupted after the stopping epoch was saved
            start_epoch = STAGE2_EPOCHS + 1

    for epoch in range(start_epoch, STAGE2_EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        t0 = time.time()

        corr_type = random.choice(CORRUPT_TYPES)
        severity  = random.randint(1, 5)

        for imgs, dens, counts, points in tqdm(train_loader,
                                       desc=f"S2 Ep {epoch}/{STAGE2_EPOCHS}",
                                       leave=False):
            imgs   = imgs.to(DEVICE)
            dens   = dens.to(DEVICE)
            counts = counts.to(DEVICE)

            # Clean forward
            pred_clean = model(imgs)

            # Corrupted forward
            with torch.no_grad():
                imgs_corr = corrupt_batch(imgs, corr_type, severity)
            pred_corrupt = model(imgs_corr)

            # Feature consistency: use MSRM output as proxy
            # (we pass clean feat = pred_clean, corrupt = pred_corrupt for simplicity)
            loss = criterion(
                pred_clean, dens, counts,
                pred_corrupt=pred_corrupt,
                pred_clean_stop=pred_clean,
                feat_clean=pred_clean,
                feat_corrupt=pred_corrupt,
                points=points,
            )

            optimizer_s2.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer_s2.step()
            epoch_loss += loss.item()

        scheduler_s2.step()
        avg_loss = epoch_loss / len(train_loader)
        val_mae, val_rmse = evaluate(val_loader, model, DEVICE)

        writer_s2.add_scalar("Loss/train", avg_loss,  epoch)
        writer_s2.add_scalar("Val/MAE",    val_mae,   epoch)
        writer_s2.add_scalar("Val/RMSE",   val_rmse,  epoch)
        writer_s2.flush()
        log_epoch(2, {"epoch": epoch, "train_loss": avg_loss, "val_mae": val_mae,
                      "val_rmse": val_rmse, "corruption": corr_type, "severity": severity},
                  history_s2)

        if val_mae < best_mae_s2:
            best_mae_s2 = val_mae
            atomic_save({"epoch": epoch, "state_dict": model.state_dict(),
                         "val_mae": val_mae}, best_ckpt_s2)

        if epoch % 10 == 0 or epoch == STAGE2_EPOCHS:
            print(f"[S2 {epoch:3d}/{STAGE2_EPOCHS}] "
                  f"loss={avg_loss:.4f}  val_MAE={val_mae:.2f}  "
                  f"val_RMSE={val_rmse:.2f}  "
                  f"corr={corr_type}/s{severity}  "
                  f"time={time.time()-t0:.1f}s")

        # full state to Drive every epoch, then the early-stopping decision
        save_stage_state(2, epoch, model, optimizer_s2, scheduler_s2, best_mae_s2, history_s2)
        stopped_early, stop_reason = early_stop_check(history_s2, S2_PATIENCE, S2_MIN_EPOCHS)
        if stopped_early:
            print(f"[S2] Early stopping at epoch {epoch}: {stop_reason}")
            break

    writer_s2.close()
    mark_stage_done(2, {"epochs": STAGE2_EPOCHS, "epochs_run": len(history_s2),
                        "stopped_early": stopped_early, "stop_reason": stop_reason, "best_val_mae": best_mae_s2, "ckpt": best_ckpt_s2})
    print(f"Stage II done after {len(history_s2)} epochs. Best val MAE: {best_mae_s2:.2f}")
elif not os.path.exists(s1_ckpt):
    print("[SKIP] Dataset or Stage I checkpoint not available.")


## 12  Adversarial Attacks (FGSM, PGD, BIM)

Used in Stage III adversarial training and robustness evaluation.


In [ ]:
import torch
import torch.nn.functional as F

# ε / α are in PIXEL units ([0,1] images), as is standard. The model sees ImageNet-normalised
# tensors, so they are converted per channel (÷ std), and adversarial images are kept inside the
# valid image range. (Previously ε was applied in normalised units, i.e. ~4.4x weaker than stated,
# and BIM clamped normalised tensors to [0,1], which destroyed the image.)
EPS   = 4 / 255   # ≈ 0.01569
ALPHA = 1 / 255   # ≈ 0.00392
STEPS = 7

_ATK_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_ATK_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def _scale(x, v):
    return v / _ATK_STD.to(x.device)          # pixel units -> normalised units, per channel


def _project(adv, imgs, eps):
    e = _scale(imgs, eps)
    adv = torch.max(torch.min(adv, imgs + e), imgs - e)                 # L_inf ball
    lo = (0 - _ATK_MEAN.to(imgs.device)) / _ATK_STD.to(imgs.device)
    hi = (1 - _ATK_MEAN.to(imgs.device)) / _ATK_STD.to(imgs.device)
    return torch.max(torch.min(adv, hi), lo).detach()                   # valid image range


def _grad(model, x, dens):
    x = x.clone().detach().requires_grad_(True)
    loss = F.mse_loss(model(x), dens)
    g, = torch.autograd.grad(loss, x)   # no gradient accumulation into the model's parameters
    return g


def fgsm_attack(model, imgs, dens, eps=EPS):
    """Fast Gradient Sign Method (untargeted, density MSE loss)."""
    model.eval()
    adv = imgs + _scale(imgs, eps) * _grad(model, imgs, dens).sign()
    return _project(adv, imgs, eps)


def pgd_attack(model, imgs, dens, eps=EPS, alpha=ALPHA, steps=STEPS):
    """Projected Gradient Descent (Madry et al.), random start."""
    model.eval()
    adv = _project(imgs + _scale(imgs, eps) * torch.empty_like(imgs).uniform_(-1, 1), imgs, eps)
    for _ in range(steps):
        adv = _project(adv + _scale(imgs, alpha) * _grad(model, adv, dens).sign(), imgs, eps)
    return adv


def bim_attack(model, imgs, dens, eps=EPS, alpha=ALPHA, steps=10):
    """Basic Iterative Method (Kurakin et al.): PGD without the random start."""
    model.eval()
    adv = imgs.clone().detach()
    for _ in range(steps):
        adv = _project(adv + _scale(imgs, alpha) * _grad(model, adv, dens).sign(), imgs, eps)
    return adv


print("Adversarial attack functions ready: FGSM, PGD, BIM")
print(f"  eps={EPS:.5f}  alpha={ALPHA:.5f}  PGD steps={STEPS}  (pixel units)")


## 13  Stage III — Adversarial Training (50 epochs)

Loads Stage II weights. Each batch alternates PGD adversarial examples.


In [ ]:
import os, time, random
import torch
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm

STAGE3_EPOCHS = 50   # upper limit (early stopping usually ends sooner)
LR_S3         = 2e-5
S3_PATIENCE   = 10
S3_MIN_EPOCHS = 10

s2_ckpt = os.path.join(CKPT_DIR, "stage2_best.pth")
best_ckpt_s3 = os.path.join(CKPT_DIR, "stage3_best.pth")

if 'train_loader' in dir() and os.path.exists(s2_ckpt) and not stage_already_done(3):
    ckpt = torch.load(s2_ckpt, map_location=DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    print(f"Loaded Stage II checkpoint (val_MAE={ckpt['val_mae']:.2f})")

    optimizer_s3 = optim.Adam(model.parameters(), lr=LR_S3, weight_decay=WEIGHT_DECAY)
    scheduler_s3 = optim.lr_scheduler.CosineAnnealingLR(
        optimizer_s3, T_max=STAGE3_EPOCHS, eta_min=LR_S3 * 0.01)
    writer_s3  = SummaryWriter(log_dir=os.path.join(LOG_DIR, "stage3"))

    best_mae_s3 = best_so_far(best_ckpt_s3)
    history_s3  = []

    start_epoch, stopped_early, stop_reason = 1, False, ""
    resumed = load_stage_state(3, model, optimizer_s3, scheduler_s3)
    if resumed:
        start_epoch, best_mae_s3, history_s3 = resumed
        stopped_early, stop_reason = early_stop_check(history_s3, S3_PATIENCE, S3_MIN_EPOCHS)
        if stopped_early:  # interrupted after the stopping epoch was saved
            start_epoch = STAGE3_EPOCHS + 1

    for epoch in range(start_epoch, STAGE3_EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        t0 = time.time()

        for imgs, dens, counts, points in tqdm(train_loader,
                                       desc=f"S3 Ep {epoch}/{STAGE3_EPOCHS}",
                                       leave=False):
            imgs, dens, counts = imgs.to(DEVICE), dens.to(DEVICE), counts.to(DEVICE)

            # Generate PGD adversarial examples
            adv_imgs = pgd_attack(model, imgs, dens)

            # Clean + adversarial combined forward
            model.train()
            pred_clean = model(imgs)
            pred_adv   = model(adv_imgs)

            loss_clean = criterion(pred_clean, dens, counts, points=points)
            loss_adv   = criterion(pred_adv,   dens, counts,
                                   pred_corrupt=pred_adv,
                                   pred_clean_stop=pred_clean, points=points)
            loss = 0.5 * loss_clean + 0.5 * loss_adv

            optimizer_s3.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer_s3.step()
            epoch_loss += loss.item()

        scheduler_s3.step()
        avg_loss = epoch_loss / len(train_loader)
        val_mae, val_rmse = evaluate(val_loader, model, DEVICE)

        writer_s3.add_scalar("Loss/train", avg_loss, epoch)
        writer_s3.add_scalar("Val/MAE",    val_mae,  epoch)
        writer_s3.add_scalar("Val/RMSE",   val_rmse, epoch)
        writer_s3.flush()
        log_epoch(3, {"epoch": epoch, "train_loss": avg_loss, "val_mae": val_mae,
                      "val_rmse": val_rmse}, history_s3)

        if val_mae < best_mae_s3:
            best_mae_s3 = val_mae
            atomic_save({"epoch": epoch, "state_dict": model.state_dict(),
                         "val_mae": val_mae}, best_ckpt_s3)

        if epoch % 5 == 0 or epoch == STAGE3_EPOCHS:
            print(f"[S3 {epoch:3d}/{STAGE3_EPOCHS}] "
                  f"loss={avg_loss:.4f}  val_MAE={val_mae:.2f}  "
                  f"val_RMSE={val_rmse:.2f}  "
                  f"time={time.time()-t0:.1f}s")

        # full state to Drive every epoch, then the early-stopping decision
        save_stage_state(3, epoch, model, optimizer_s3, scheduler_s3, best_mae_s3, history_s3)
        stopped_early, stop_reason = early_stop_check(history_s3, S3_PATIENCE, S3_MIN_EPOCHS)
        if stopped_early:
            print(f"[S3] Early stopping at epoch {epoch}: {stop_reason}")
            break

    writer_s3.close()
    mark_stage_done(3, {"epochs": STAGE3_EPOCHS, "epochs_run": len(history_s3),
                        "stopped_early": stopped_early, "stop_reason": stop_reason, "best_val_mae": best_mae_s3, "ckpt": best_ckpt_s3})
    print(f"Stage III done after {len(history_s3)} epochs. Best val MAE: {best_mae_s3:.2f}")
elif not os.path.exists(s2_ckpt):
    print("[SKIP] Dataset or Stage II checkpoint not available.")


## 14  Load Best Checkpoint for Evaluation

In [ ]:
import os
import torch

def load_best_checkpoint(model, ckpt_dir):
    """Load the best available checkpoint (Stage 3 > 2 > 1)."""
    for stage in [3, 2, 1]:
        path = os.path.join(ckpt_dir, f"stage{stage}_best.pth")
        if os.path.exists(path):
            ckpt = torch.load(path, map_location=DEVICE)
            model.load_state_dict(ckpt["state_dict"])
            print(f"Loaded Stage {stage} checkpoint  val_MAE={ckpt.get('val_mae', '?'):.2f}")
            return stage
    print("[WARNING] No checkpoint found — using randomly initialised weights.")
    return 0

loaded_stage = load_best_checkpoint(model, CKPT_DIR)
model.eval()
update_summary("evaluated_checkpoint", {"stage": loaded_stage})


## 15  Clean Test-Set Evaluation

Reports MAE, RMSE, NAE, and GAME-0 to GAME-3 (GAME-L: the image is split into 4^L regions and the absolute count errors of the regions are summed; region GT counts come from the GT density map). Per-image predictions and all metrics are saved to Drive.


In [ ]:
import numpy as np
import pandas as pd
import torch
from tqdm.notebook import tqdm


def game_metric(pred_map, gt_map, level):
    """Grid Average Mean absolute Error (Guerrero-Gomez-Olmedo et al., 2015) for one image.
    pred_map, gt_map: (h, w) density maps that sum to the counts."""
    n = 2 ** level
    rows = np.array_split(np.arange(gt_map.shape[0]), n)
    cols = np.array_split(np.arange(gt_map.shape[1]), n)
    return float(sum(abs(pred_map[np.ix_(r, c)].sum() - gt_map[np.ix_(r, c)].sum())
                     for r in rows for c in cols))


def evaluate_full(loader, net, device, return_per_image=False):
    net.eval()
    maes, rmses, naes, per_image = [], [], [], []
    game = {0: [], 1: [], 2: [], 3: []}

    with torch.no_grad():
        for i, (imgs, dens, counts) in enumerate(tqdm(loader, desc="Evaluating", leave=False)):
            imgs = imgs.to(device)
            pred = net(imgs)
            p_cnt = pred.sum().item()
            g_cnt = counts.item()

            maes.append(abs(p_cnt - g_cnt))
            rmses.append((p_cnt - g_cnt) ** 2)
            if g_cnt > 0:
                naes.append(abs(p_cnt - g_cnt) / g_cnt)
            pred_map = pred.squeeze().cpu().numpy()
            gt_map = dens.squeeze().numpy()
            game[0].append(abs(p_cnt - g_cnt))  # GAME-0 is the MAE, from the exact GT count
            for lvl in (1, 2, 3):
                game[lvl].append(game_metric(pred_map, gt_map, lvl))
            per_image.append({"index": i, "gt_count": g_cnt, "pred_count": p_cnt,
                              "abs_error": abs(p_cnt - g_cnt)})

    mae  = float(np.mean(maes))
    rmse = float(np.sqrt(np.mean(rmses)))
    nae  = float(np.mean(naes)) if naes else float("nan")
    games = {k: float(np.mean(v)) for k, v in game.items()}
    if return_per_image:
        return mae, rmse, nae, games, per_image
    return mae, rmse, nae, games


if 'test_loader' in dir():
    mae, rmse, nae, games, per_image = evaluate_full(test_loader, model, DEVICE, return_per_image=True)
    print("=" * 50)
    print(f"  MAE   : {mae:.2f}")
    print(f"  RMSE  : {rmse:.2f}")
    print(f"  NAE   : {nae:.4f}")
    for k, v in games.items():
        print(f"  GAME-{k}: {v:.2f}")
    print("=" * 50)

    per_image_df = pd.DataFrame(per_image)
    per_image_df.insert(1, "image", [os.path.basename(p) for p in test_imgs])
    save_csv(per_image_df, "clean_test_per_image.csv", index=False)
    clean_metrics = {"MAE": mae, "RMSE": rmse, "NAE": nae,
                     **{f"GAME-{k}": v for k, v in games.items()},
                     "checkpoint_stage": loaded_stage, "n_images": len(per_image)}
    save_json(clean_metrics, "clean_test_metrics.json")
    update_summary("clean_test", clean_metrics)
else:
    print("[SKIP] test_loader not available.")


## 16  Corruption Robustness Evaluation

Measures MAE on all 12 corruption types × 5 severity levels and computes the
Robustness Degradation (RD) metric: RD = (MAE_corrupt − MAE_clean) / MAE_clean × 100.


In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

def evaluate_corruption(loader, net, device, corruption_type, severity):
    net.eval()
    maes = []
    with torch.no_grad():
        for imgs, dens, counts in loader:
            imgs = imgs.to(device)
            imgs_c = corrupt_batch(imgs, corruption_type, severity)
            pred   = net(imgs_c)
            p_cnt  = pred.sum().item()
            g_cnt  = counts.item()
            maes.append(abs(p_cnt - g_cnt))
    return float(np.mean(maes))

if 'test_loader' in dir():
    # Clean baseline
    mae_clean, _, _, _ = evaluate_full(test_loader, model, DEVICE)
    print(f"Clean MAE: {mae_clean:.2f}")

    results = {}
    for ct in tqdm(list(SEVERITY_PARAMS.keys()), desc="Corruption types"):
        results[ct] = []
        for sev in range(1, 6):
            m = evaluate_corruption(test_loader, model, DEVICE, ct, sev)
            results[ct].append(m)

    # Build DataFrame
    df = pd.DataFrame(results,
                      index=[f"Sev-{i}" for i in range(1, 6)]).T
    df["Mean_MAE"] = df.mean(axis=1)
    df["RD_%"]     = ((df["Mean_MAE"] - mae_clean) / mae_clean * 100).round(2)
    print("\nCorruption Robustness Results:")
    print(df.to_string())

    # Heatmap
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.heatmap(df.iloc[:, :5].astype(float), annot=True, fmt=".1f",
                cmap="YlOrRd", ax=ax)
    ax.set_title("Corruption MAE Heatmap (12 types × 5 severities)")
    ax.set_xlabel("Severity")
    ax.set_ylabel("Corruption type")
    plt.tight_layout()
    save_fig(fig, "corruption_heatmap.png")
    plt.show()

    # RD bar chart
    fig2, ax2 = plt.subplots(figsize=(12, 4))
    df["RD_%"].plot(kind="bar", ax=ax2, color="steelblue", edgecolor="black")
    ax2.axhline(0, color="red", ls="--", lw=1)
    ax2.set_title("Robustness Degradation (RD%) per Corruption Type")
    ax2.set_ylabel("RD (%)")
    ax2.set_xlabel("Corruption type")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    save_fig(fig2, "corruption_rd_bar.png")
    plt.show()

    save_csv(df, "corruption_results.csv")
    update_summary("corruption", {"clean_MAE": mae_clean,
                                  "mean_corrupt_MAE": float(df["Mean_MAE"].mean()),
                                  "mean_RD_%": float(df["RD_%"].mean())})
else:
    print("[SKIP] test_loader not available.")


## 17  Adversarial Robustness Evaluation (FGSM, PGD, BIM)

Evaluates MAE and count error under three white-box attacks at multiple ε values.


In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

EPS_LIST = [1/255, 2/255, 4/255, 8/255, 16/255]

def eval_adversarial(loader, net, device, attack_fn, **kwargs):
    net.eval()
    maes = []
    for imgs, dens, counts in tqdm(loader, desc="Adv eval", leave=False):
        imgs, dens, counts = imgs.to(device), dens.to(device), counts.to(device)
        adv = attack_fn(net, imgs, dens, **kwargs)
        with torch.no_grad():
            pred  = net(adv)
            p_cnt = pred.sum().item()
            g_cnt = counts.item()
            maes.append(abs(p_cnt - g_cnt))
    return float(np.mean(maes))

if 'test_loader' in dir():
    records = []
    for eps in tqdm(EPS_LIST, desc="ε sweep"):
        row = {"epsilon": f"{eps*255:.0f}/255"}
        row["FGSM"] = eval_adversarial(test_loader, model, DEVICE, fgsm_attack, eps=eps)
        row["PGD"]  = eval_adversarial(test_loader, model, DEVICE, pgd_attack,
                                        eps=eps, alpha=eps/4, steps=STEPS)
        row["BIM"]  = eval_adversarial(test_loader, model, DEVICE, bim_attack,
                                        eps=eps, alpha=eps/10, steps=10)
        records.append(row)

    adv_df = pd.DataFrame(records).set_index("epsilon")
    print("\nAdversarial Robustness — MAE:")
    print(adv_df.to_string())

    # Line plot
    fig, ax = plt.subplots(figsize=(8, 5))
    for col in ["FGSM", "PGD", "BIM"]:
        ax.plot(adv_df.index, adv_df[col], marker="o", label=col)
    ax.axhline(mae_clean if 'mae_clean' in dir() else 0,
               ls="--", color="green", label="Clean MAE")
    ax.set_title("MAE vs Perturbation Strength")
    ax.set_xlabel("ε (× 255)")
    ax.set_ylabel("MAE")
    ax.legend()
    plt.tight_layout()
    save_fig(fig, "adversarial_robustness.png")
    plt.show()

    save_csv(adv_df, "adversarial_results.csv")
    update_summary("adversarial", {f"{atk}@{eps}": float(adv_df.loc[eps, atk])
                                   for eps in adv_df.index for atk in ["FGSM", "PGD", "BIM"]})
else:
    print("[SKIP] test_loader not available.")


## 18  XAI Visualisations

### 18.1 Gradient × Input Saliency
Justified over Grad-CAM for regression tasks (density prediction):
the saliency map S = |∇_x L| ⊙ |x| indicates pixel-level contribution
to the predicted density without class-discriminative assumptions.


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

MEAN_T = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
STD_T  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def gradient_x_input_saliency(net, img_tensor):
    """
    Compute gradient × input saliency for density prediction.
    img_tensor : (1, 3, H, W) normalised, on device.
    Returns    : (H, W) numpy array in [0, 1].
    """
    net.eval()
    # move to the device FIRST, then mark as requiring grad, so x is a leaf and x.grad is filled
    x = img_tensor.clone().detach().to(DEVICE).requires_grad_(True)
    pred = net(x)
    score = pred.sum()
    net.zero_grad()
    score.backward()
    saliency = (x.grad.abs() * x.detach().abs()).squeeze(0)  # (3, H, W)
    saliency = saliency.max(dim=0).values            # (H, W)
    saliency = saliency.cpu().numpy()
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
    return saliency


def visualise_sample(net, img_tensor, gt_density, sample_idx=0):
    img_np = (img_tensor.squeeze(0).cpu() * STD_T + MEAN_T).clamp(0,1)
    img_np = img_np.numpy().transpose(1,2,0)

    # Saliency
    sal = gradient_x_input_saliency(net, img_tensor)

    # Predicted density
    net.eval()
    with torch.no_grad():
        pred_den = net(img_tensor.to(DEVICE)).squeeze().cpu().numpy()

    # Ground-truth density
    gt_den = gt_density.squeeze().numpy()

    # Error map
    pred_up = F.interpolate(
        torch.from_numpy(pred_den).unsqueeze(0).unsqueeze(0),
        size=gt_den.shape, mode="bilinear", align_corners=False
    ).squeeze().numpy()
    err_map = np.abs(pred_up - gt_den)

    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    titles = ["Input Image", "GT Density", "Pred Density",
              "Error Map", "Gradient Saliency"]
    data   = [img_np, gt_den, pred_den, err_map, sal]
    cmaps  = [None, "hot", "hot", "bwr", "plasma"]

    for ax, title, d, cmap in zip(axes, titles, data, cmaps):
        if cmap is None:
            ax.imshow(d)
        else:
            ax.imshow(d, cmap=cmap)
        ax.set_title(title, fontsize=11)
        ax.axis("off")
    plt.suptitle(f"XAI Visualisation — Sample {sample_idx}", fontsize=13)
    plt.tight_layout()
    save_fig(fig, f"xai_sample_{sample_idx}.png")
    plt.show()
    p_count = pred_den.sum()
    g_count = gt_den.sum()
    print(f"  Predicted count: {p_count:.1f}  |  GT count: {g_count:.1f}")


if 'test_loader' in dir():
    it = iter(test_loader)
    for idx in range(min(3, len(test_loader))):
        imgs, dens, counts = next(it)
        visualise_sample(model, imgs, dens, sample_idx=idx)
else:
    print("[SKIP] test_loader not available.")


### 18.2  DAA Reliability Gate Visualisation

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

def get_daa_reliability_map(net, img_tensor):
    """
    Extract the DAA reliability gate activation for a single image.
    Uses a forward hook on net.daa.gate.
    Returns: (H, W) numpy array aggregated over channels.
    """
    activations = {}

    def hook_fn(module, inp, out):
        activations["gate"] = out.detach().cpu()

    handle = net.daa.gate.register_forward_hook(hook_fn)
    net.eval()
    with torch.no_grad():
        _ = net(img_tensor.to(DEVICE))
    handle.remove()

    gate = activations["gate"].squeeze(0)  # (C, H, W)
    reliability = gate.mean(dim=0).numpy()  # (H, W)
    reliability = (reliability - reliability.min()) / (reliability.max() - reliability.min() + 1e-8)
    return reliability


if 'test_loader' in dir():
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    it = iter(test_loader)
    for col in range(3):
        imgs, dens, counts = next(it)
        img_np = (imgs.squeeze(0) * STD_T + MEAN_T).clamp(0,1).numpy().transpose(1,2,0)
        rel    = get_daa_reliability_map(model, imgs)

        axes[0, col].imshow(img_np)
        axes[0, col].set_title(f"Input {col+1}")
        axes[0, col].axis("off")

        im = axes[1, col].imshow(rel, cmap="viridis")
        axes[1, col].set_title(f"DAA Reliability Gate {col+1}")
        axes[1, col].axis("off")
        plt.colorbar(im, ax=axes[1, col], fraction=0.046, pad=0.04)

    plt.suptitle("DAA Reliability Gate Activations", fontsize=13)
    plt.tight_layout()
    save_fig(fig, "daa_reliability_maps.png")
    plt.show()
else:
    print("[SKIP] test_loader not available.")


## 19  Ablation Study

Systematically removes RMSC-Net components and reports MAE/RMSE degradation
to quantify each module's contribution.


In [ ]:
import copy, torch, numpy as np
from tqdm.notebook import tqdm

def ablate_model(variant: str) -> torch.nn.Module:
    """Return a modified RMSCNet variant for ablation."""
    m = RMSCNet(**MODEL_KWARGS).to(DEVICE)
    # Load best weights as starting point (components removed after init)
    load_best_checkpoint(m, CKPT_DIR)

    if variant == "no_msrm":
        # Replace MSRM with identity
        m.msrm = torch.nn.Identity()

    elif variant == "no_daa":
        # Replace DAA with identity
        m.daa = torch.nn.Identity()

    elif variant == "no_consistency":
        # Model identical; we just won't use consistency loss during eval
        pass  # ablated at training time

    elif variant == "no_fusion_adapt":
        # Switch adaptive fusion to uniform average
        m.fusion.mode = "uniform"

    elif variant == "no_stage3":
        # Load Stage 2 checkpoint instead
        p = os.path.join(CKPT_DIR, "stage2_best.pth")
        if os.path.exists(p):
            ckpt = torch.load(p, map_location=DEVICE)
            m.load_state_dict(ckpt["state_dict"])

    m.eval()
    return m

ABLATION_VARIANTS = {
    "Full RMSC-Net":       "full",
    "w/o MSRM":            "no_msrm",
    "w/o DAA":             "no_daa",
    "w/o Adaptive Fusion": "no_fusion_adapt",
    "w/o Stage-III (AT)":  "no_stage3",
}

if 'test_loader' in dir():
    ablation_results = {}
    for name, variant in ABLATION_VARIANTS.items():
        if variant == "full":
            m = model
        else:
            m = ablate_model(variant)
        mae, rmse, nae, _ = evaluate_full(test_loader, m, DEVICE)
        ablation_results[name] = {"MAE": round(mae, 2), "RMSE": round(rmse, 2)}
        print(f"  {name:<30}  MAE={mae:.2f}  RMSE={rmse:.2f}")

    import pandas as pd
    abl_df = pd.DataFrame(ablation_results).T
    print("\nAblation Summary:")
    print(abl_df.to_string())
    save_csv(abl_df, "ablation_results.csv")
    update_summary("ablation", {k: v for k, v in ablation_results.items()})

    # Bar plot
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 4))
    abl_df["MAE"].plot(kind="bar", ax=ax, color="cornflowerblue", edgecolor="black")
    ax.set_title("Ablation Study — MAE on ShanghaiTech Part A Test Set")
    ax.set_ylabel("MAE")
    ax.set_xlabel("Model variant")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    save_fig(fig, "ablation_bar.png")
    plt.show()
else:
    print("[SKIP] test_loader not available.")


## 20  Statistical Validation

Paired t-test, Wilcoxon signed-rank test, and 95% bootstrap confidence intervals
for MAE differences between RMSC-Net and its ablated variants.


In [ ]:
import numpy as np
from scipy import stats

def bootstrap_ci(errors_a, errors_b, n_bootstrap=10000, ci=0.95):
    """Bootstrap 95% CI on the mean MAE improvement (a - b)."""
    diffs = np.array(errors_a) - np.array(errors_b)
    boot  = np.array([
        np.random.choice(diffs, len(diffs), replace=True).mean()
        for _ in range(n_bootstrap)
    ])
    lo = np.percentile(boot, (1 - ci) / 2 * 100)
    hi = np.percentile(boot, (1 + ci) / 2 * 100)
    return lo, hi


def collect_per_image_errors(loader, net, device):
    """Return list of per-image absolute errors."""
    net.eval()
    errs = []
    with torch.no_grad():
        for imgs, dens, counts in loader:
            imgs = imgs.to(device)
            pred = net(imgs)
            p_cnt = pred.sum().item()
            g_cnt = counts.item()
            errs.append(abs(p_cnt - g_cnt))
    return errs


if 'test_loader' in dir():
    import pandas as pd
    errs_full = collect_per_image_errors(test_loader, model, DEVICE)
    stat_rows = []

    print("Statistical tests vs ablated variants:")
    for name, variant in list(ABLATION_VARIANTS.items())[1:]:
        abl = ablate_model(variant)
        errs_abl = collect_per_image_errors(test_loader, abl, DEVICE)

        t_stat, t_p   = stats.ttest_rel(errs_full, errs_abl)
        w_stat, w_p   = stats.wilcoxon(errs_full, errs_abl)
        ci_lo, ci_hi  = bootstrap_ci(errs_full, errs_abl)

        print(f"\n  {name}")
        print(f"    Paired t-test  : t={t_stat:.3f},  p={t_p:.4f}")
        print(f"    Wilcoxon       : W={w_stat:.1f},   p={w_p:.4f}")
        print(f"    Bootstrap 95%CI: [{ci_lo:.2f}, {ci_hi:.2f}]")
        sig = "(significant)" if t_p < 0.05 else "(not significant)"
        print(f"    → {sig} at α=0.05")
        stat_rows.append({"variant": name, "MAE_full": float(np.mean(errs_full)),
                          "MAE_variant": float(np.mean(errs_abl)),
                          "t": t_stat, "t_p": t_p, "wilcoxon_W": w_stat, "wilcoxon_p": w_p,
                          "boot_ci_low": ci_lo, "boot_ci_high": ci_hi,
                          "significant_0.05": bool(t_p < 0.05)})
    save_csv(pd.DataFrame(stat_rows), "statistical_tests.csv", index=False)
else:
    print("[SKIP] test_loader not available.")


## 21  TensorBoard Training Curves

In [ ]:
# Load TensorBoard in Colab
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}


## 22  Results Dashboard & Summary Report

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import numpy as np

# Collect all saved result CSVs
result_files = {
    "Corruption": os.path.join(RESULT_DIR, "corruption_results.csv"),
    "Adversarial": os.path.join(RESULT_DIR, "adversarial_results.csv"),
    "Ablation":   os.path.join(RESULT_DIR, "ablation_results.csv"),
}

for name, path in result_files.items():
    if os.path.exists(path):
        df = pd.read_csv(path, index_col=0)
        print(f"\n{'='*55}")
        print(f"  {name} Results")
        print('='*55)
        print(df.to_string())
    else:
        print(f"[{name}] result file not found at {path}")

# Dashboard figure
fig_paths = [
    (os.path.join(FIG_DIR, "corruption_heatmap.png"), "Corruption Heatmap"),
    (os.path.join(FIG_DIR, "corruption_rd_bar.png"),  "RD Bars"),
    (os.path.join(FIG_DIR, "adversarial_robustness.png"), "Adversarial MAE"),
    (os.path.join(FIG_DIR, "ablation_bar.png"),        "Ablation MAE"),
]

available = [(p, t) for p, t in fig_paths if os.path.exists(p)]
if available:
    n = len(available)
    fig, axes = plt.subplots(1, n, figsize=(6*n, 5))
    if n == 1:
        axes = [axes]
    for ax, (p, t) in zip(axes, available):
        ax.imshow(np.array(Image.open(p)))
        ax.set_title(t, fontsize=12)
        ax.axis("off")
    plt.suptitle("RMSC-Net Evaluation Dashboard", fontsize=15)
    plt.tight_layout()
    save_fig(fig, "dashboard.png")
    plt.show()
else:
    print("[INFO] No result images available yet — run evaluation cells first.")

if os.path.exists(SUMMARY_PATH):
    print("\nresults/summary.json:")
    print(open(SUMMARY_PATH).read())


## 23  Checkpoint Save / Restore Utilities

Use these helpers to save a mid-training snapshot or reload a specific
checkpoint for further fine-tuning or evaluation.


In [ ]:
import torch, os, glob

def save_checkpoint(model, optimizer, epoch, val_mae, tag="manual"):
    path = os.path.join(CKPT_DIR, f"ckpt_{tag}_ep{epoch:03d}_mae{val_mae:.2f}.pth")
    torch.save({
        "epoch":       epoch,
        "state_dict":  model.state_dict(),
        "optimizer":   optimizer.state_dict(),
        "val_mae":     val_mae,
    }, path)
    print(f"Saved checkpoint → {path}")
    return path


def list_checkpoints():
    files = sorted(glob.glob(os.path.join(CKPT_DIR, "*.pth")))
    print(f"Checkpoints in {CKPT_DIR}:")
    for i, f in enumerate(files):
        # weights_only=False: stageN_last.pth also holds history/RNG state (our own files)
        ckpt = torch.load(f, map_location="cpu", weights_only=False)
        print(f"  [{i}] {os.path.basename(f)}  "
              f"epoch={ckpt.get('epoch','?')}  "
              f"val_MAE={ckpt.get('val_mae', ckpt.get('best_mae', '?'))}")
    return files


def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["state_dict"])
    if optimizer and "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    print(f"Loaded {os.path.basename(path)}  "
          f"epoch={ckpt.get('epoch','?')}  val_MAE={ckpt.get('val_mae','?')}")
    return ckpt.get("epoch", 0)


# Demo: list available checkpoints
list_checkpoints()


## 24  Model Export

Export the trained model to ONNX and TorchScript for deployment.


In [ ]:
import torch
import os

os.makedirs(EXPORT_DIR, exist_ok=True)  # defined in Section 2 (on Drive)

model.eval()
dummy_input = torch.randn(1, 3, 512, 512).to(DEVICE)

# ── TorchScript ────────────────────────────────────────────────────────────
try:
    scripted = torch.jit.trace(model, dummy_input)
    ts_path  = os.path.join(EXPORT_DIR, "rmscnet_scripted.pt")
    scripted.save(ts_path)
    print(f"TorchScript saved → {ts_path}")
except Exception as e:
    print(f"TorchScript export failed: {e}")

# ── ONNX ──────────────────────────────────────────────────────────────────
try:
    onnx_path = os.path.join(EXPORT_DIR, "rmscnet.onnx")
    torch.onnx.export(
        model, dummy_input, onnx_path,
        opset_version=17,
        input_names=["image"],
        output_names=["density_map"],
        dynamic_axes={"image": {0: "batch", 2: "H", 3: "W"},
                      "density_map": {0: "batch"}},
        verbose=False,
    )
    print(f"ONNX model saved → {onnx_path}")

    # Verify with onnxruntime
    try:
        import onnxruntime as ort
        sess = ort.InferenceSession(onnx_path,
               providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
        dummy_np = dummy_input.cpu().numpy()
        out = sess.run(None, {"image": dummy_np})
        print(f"ONNX verification passed — output shape: {out[0].shape}")
    except ImportError:
        print("onnxruntime not installed — skipping ONNX verification.")
except Exception as e:
    print(f"ONNX export failed: {e}")


## 25  Quick Inference Demo

Run inference on a single image (upload your own or use a test image).


In [ ]:
from google.colab import files
import io
from PIL import Image
import torch
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np

def infer_single_image(image_path_or_pil, model, device):
    """End-to-end inference on a single image."""
    if isinstance(image_path_or_pil, str):
        img = Image.open(image_path_or_pil).convert("RGB")
    else:
        img = image_path_or_pil.convert("RGB")

    transform = T.Compose([
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    w, h = img.size
    img_t = transform(pad_image(img)).unsqueeze(0).to(device)  # pad to a multiple of 16

    model.eval()
    with torch.no_grad():
        density = model(img_t)

    density_np = density.squeeze().cpu().numpy()[:h // DENSITY_SCALE, :w // DENSITY_SCALE]
    count      = float(density.sum().item())

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(np.array(img))
    axes[0].set_title("Input Image")
    axes[0].axis("off")

    axes[1].imshow(density_np, cmap="hot")
    axes[1].set_title(f"Predicted Density Map  (Count ≈ {count:.0f})")
    axes[1].axis("off")
    plt.colorbar(axes[1].images[0], ax=axes[1], fraction=0.046, pad=0.04)
    plt.tight_layout()
    name = os.path.splitext(os.path.basename(image_path_or_pil))[0] \
        if isinstance(image_path_or_pil, str) else "uploaded"
    save_fig(fig, f"inference_{name}.png")
    plt.show()

    print(f"Estimated crowd count: {count:.1f}")
    return count


# Upload an image for demonstration
print("Upload a crowd image (JPG/PNG) to run inference:")
print("  Option A: upload via the cell below")
print("  Option B: use a test image if the dataset is loaded")
print()

if 'test_imgs' in dir() and len(test_imgs) > 0:
    # Use the first test image as demo
    count = infer_single_image(test_imgs[0], model, DEVICE)
else:
    print("Run: uploaded = files.upload() then pass the file to infer_single_image()")
    print("Example:")
    print("  uploaded = files.upload()")
    print("  path = list(uploaded.keys())[0]")
    print("  infer_single_image(path, model, DEVICE)")


## 26  Notes & Citations

### Architecture Reference
```
Input (B,3,H,W)
  ↓  Encoder: ImageNet-pretrained MobileNetV2 (F1–F4 at strides 4/8/16/32, 1x1-projected)
     [or ENCODER="scratch": RobustStem + LightweightEncoder, randomly initialised]
  ↓  MSRM (dilated DWConv d=1,2,3,5 on F3 + residual fusion)
  ↓  DAA  (Channel Attention → Spatial Attention → Reliability Gate)
  ↓  RobustFusion (adaptive softmax-weighted multi-scale fusion)
  ↓  RobustFusion aligns F1–F4 at F2 (1/8 resolution)
  ↓  DensityDecoder (3 DS-conv stages, 256→128→64→32, no upsampling)
  ↓  Density Map (B,1,H/8,W/8)
```

### Training Curriculum
| Stage | Epochs | Data                     | Loss terms                               |
|-------|--------|--------------------------|------------------------------------------|
| I     | 100    | Clean                    | L_density + L_count                      |
| II    | 100    | Clean + Corrupted        | + L_consistency + L_feature              |
| III   | 50     | Clean + Corrupted + Adv  | + adversarial consistency                |

### Loss Coefficients
λ_d = 1.0,  λ_c = 0.1,  λ_cons = 0.5,  λ_f = 0.1

### Cite this work
```
@article{rmscnet2025,
  title   = {{RMSC-Net}: A Lightweight Robust Multi-Scale Consistency Network
             for Dense Crowd Counting under Environmental and Adversarial Perturbations},
  author  = {[AUTHORS]},
  journal = {[JOURNAL]},
  year    = {2025},
}
```

### References (key works)
- Guo et al., DADNet, ACM MM 2019 — multi-scale density estimation
- Huang et al., Counting Crowds in Bad Weather, ICCV 2023 — environmental robustness
- Xiao et al., Adversarial patch attacks, KNOSYS 2025 — adversarial robustness
- Wang et al., Efficient crowd counting via dual knowledge distillation, TIP 2023
- Yuan et al., Crowd counting under haze+rain, ICME 2024

See `references.bib` in the manuscript folder for the full BibTeX bibliography.
